In [4]:
import pyautogui
import time

time.sleep(5)
while True:
    pyautogui.click() 
    time.sleep(0.07)

KeyboardInterrupt: 

In [6]:
import webbrowser

webbrowser.open("https://popcat.click")

True

In [7]:
# Hand Gesture Detection using Meidapipe

In [ ]:
import cv2 as cv
import mediapipe as mp
import numpy as np
import webbrowser
from pycaw.pycaw import AudioUtilities, IAudioEndpointVolume #controlling audio device on Windows systems
from ctypes import cast, POINTER
from comtypes import CLSCTX_ALL
import pyautogui # controlling mouse &b keyboard
import time

BaseOptions = mp.tasks.BaseOptions
GestureRecognizer = mp.tasks.vision.GestureRecognizer
GestureRecognizerOptions = mp.tasks.vision.GestureRecognizerOptions
GestureRecognizerResult = mp.tasks.vision.GestureRecognizerResult

VisionRunningMode = mp.tasks.vision.RunningMode

last_result = None

def print_result(result: GestureRecognizerResult, output_image: mp.Image, timestamp_ms: int):
    global last_result
    last_result = result

options = GestureRecognizerOptions(
    base_options=BaseOptions(model_asset_path='gesture_recognizer.task'),
    running_mode=VisionRunningMode.LIVE_STREAM,
    result_callback=print_result
)

# Set up Windows volume control
devices = AudioUtilities.GetSpeakers()
interface = devices.Activate(IAudioEndpointVolume._iid_, CLSCTX_ALL, None)
volume = cast(interface, POINTER(IAudioEndpointVolume))
vol_min, vol_max = volume.GetVolumeRange()[:2]

opened_youtube = False # controller Velocity
opened_popcat = False
cap = cv.VideoCapture(0)

with GestureRecognizer.create_from_options(options) as recognizer:
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            continue

        frame = cv.flip(frame, 1)
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        timestamp = int(cv.getTickCount() / cv.getTickFrequency() * 1000)
        recognizer.recognize_async(mp_image, timestamp)

        if last_result and last_result.hand_landmarks:
            for hand_landmarks in last_result.hand_landmarks:
                landmarks = []
                for idx, landmark in enumerate(hand_landmarks):
                    x = int(landmark.x * frame.shape[1])
                    y = int(landmark.y * frame.shape[0])
                    landmarks.append((x, y))
                    cv.circle(frame, (x, y), 5, (0, 255, 0), -1)
                    cv.putText(frame, str(idx), (x + 5, y - 5), cv.FONT_HERSHEY_SIMPLEX, 0.4, (255, 0, 0), 1)

                connection_pairs = [(0,1), (1,2), (2,3), (3,4), (5,6), (6,7), (7,8),
                                    (9,10), (10,11), (11,12), (13,14), (14,15), (15,16),
                                    (17,18), (18,19), (19,20)]
                for start, end in connection_pairs:
                    if start < len(landmarks) and end < len(landmarks):
                        cv.line(frame, landmarks[start], landmarks[end], (0, 255, 255), 2)

        if last_result and last_result.gestures:
            gesture = last_result.gestures[0][0].category_name
            score = last_result.gestures[0][0].score
            cv.putText(frame, f'Gesture: {gesture} ({score:.2f})', (10, 30),
                        cv.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2)

            current_vol = volume.GetMasterVolumeLevel()
            print
            if gesture == "Thumb_Up":
                volume.SetMasterVolumeLevel(min(current_vol + 1.0, vol_max), None)
            elif gesture == "Thumb_Down":
                volume.SetMasterVolumeLevel(max(current_vol - 1.0, vol_min), None)
            elif gesture == "Victory":
                # webbrowser.open("https://www.youtube.com")
                # opened_youtube = True
                time.sleep(5)
                while True:
                    pyautogui.click() 
                    time.sleep(0.07)
            elif gesture == "Open_Plam" and not opened_popcat:
                webbrowser.open("https://popcat.click/")
                opened_popcat = True
                subprocess.Popen("osk.exe")

        cv.imshow('Gesture Control', frame)
        if cv.waitKey(5) & 0xFF == 27:
            break

cap.release()
cv.destroyAllWindows()

